In [0]:
from pyspark.sql import functions as F

bucket = "de-e2e-413612133697-ap-southeast-1-an"

manifest_root = f"s3://{bucket}/lakehouse/landing/douyin/media_manifest/json/"
bronze_manifest_path = f"s3://{bucket}/lakehouse/bronze/douyin/media_manifest_raw_delta/"

raw_text_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .text(manifest_root)
    .select(
        F.col("_metadata.file_path").alias("source_file"),
        F.col("value").alias("raw_json_string"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
)

parsed_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(manifest_root)
    .select(
        F.col("_metadata.file_path").alias("source_file"),
        F.col("pipeline"),
        F.col("source"),
        F.col("zone"),
        F.col("artifact"),
        F.col("niche"),
        F.col("account_id"),
        F.col("generated_at"),
        F.struct("*").alias("raw_struct")
    )
)

bronze_manifest_df = raw_text_df.join(parsed_df, on="source_file", how="inner")

display(bronze_manifest_df)

In [0]:
(
    bronze_manifest_df.write
    .format("delta")
    .mode("append")
    .save(bronze_manifest_path)
)

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS de_e2e.bronze.douyin_media_manifest_raw
            USING DELTA
            LOCATION '{bronze_manifest_path}'
          """)

In [0]:
%sql
SELECT source_file, account_id, artifact
FROM de_e2e.bronze.douyin_media_manifest_raw
LIMIT 20;